# Production Candidate v1 — MTF Microstructure Strategy

Notebook ini adalah versi **siap uji** yang disusun dari hasil robustness. Fokusnya bukan mencari curve terbaik, tetapi membangun kandidat strategi yang:
- sederhana
- jujur
- modular
- mudah difalsifikasi
- siap untuk forward test berikutnya

**Default kandidat v1**
- H1 context
- M15 signal / execution
- one-position-at-a-time
- families: `sweep_reclaim`, `rejection`, `compression_expansion`
- `momentum_break` dinonaktifkan
- sessions: `london`, `ny_overlap`
- TP = `2.0R`
- timeout = `12`
- swing lookback = `2`
- spread/slippage = `0` **hanya untuk baseline riset**

> Catatan penting: dari hasil robustness sebelumnya, edge baseline tetap hidup, tetapi **adverse execution** merusak performa secara material. Jadi notebook ini **siap uji**, namun belum siap live tanpa kalibrasi cost broker.


In [ ]:
# ===== 0. Imports =====
from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

In [ ]:
# ===== 1. Config =====
@dataclass
class Config:
    data_path: str = "../../data/raw/XAUUSD_M1.csv"
    output_dir: str = "outputs_production_v1"

    source_tf: str = "M1"
    context_tf: str = "H1"
    signal_tf: str = "M15"
    execution_tf: str = "M15"

    enabled_families: Tuple[str, ...] = ("sweep_reclaim", "rejection", "compression_expansion")
    enabled_sessions: Tuple[str, ...] = ("london", "ny_overlap")

    tp_r_multiple: float = 2.0
    timeout_bars: int = 12
    swing_lookback: int = 2
    rolling_level_n: int = 20

    one_position_at_a_time: bool = True
    use_atr_floor_cap: bool = False
    atr_period: int = 14
    atr_floor_mult: Optional[float] = None
    atr_cap_mult: Optional[float] = None

    spread_points: float = 0.0
    slippage_points: float = 0.0
    entry_delay_bars: int = 0

    train_ratio: float = 0.6
    val_ratio: float = 0.2

CFG = Config()
OUTPUT_DIR = Path(CFG.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(CFG)

## Data contract
Loader diasumsikan mengikuti format CSV yang selama ini kamu pakai:
- kolom: `date,time,open,high,low,close,volume`
- skip 1 row header
- parse waktu UTC
- urut ascending


In [ ]:
# ===== 2. Data loading & QC =====
def load_ohlcv(path: str) -> pd.DataFrame:
    names = ["date", "time", "open", "high", "low", "close", "volume"]
    df = pd.read_csv(path, skiprows=1, names=names)
    dt = pd.to_datetime(df["date"] + " " + df["time"], format="%Y.%m.%d %H:%M", utc=True, errors="coerce")
    df = df.assign(timestamp=dt).dropna(subset=["timestamp"]).drop(columns=["date", "time"])
    df = df.set_index("timestamp").sort_index()
    df = df[~df.index.duplicated(keep="first")]
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["open", "high", "low", "close"])
    return df

def qc_report(df: pd.DataFrame) -> Dict:
    full_index = pd.date_range(df.index.min(), df.index.max(), freq="1min", tz="UTC")
    missing = full_index.difference(df.index)
    rep = {
        "rows": int(len(df)),
        "start": str(df.index.min()),
        "end": str(df.index.max()),
        "duplicate_index": int(df.index.duplicated().sum()),
        "missing_1m_gaps": int(len(missing)),
        "nonpositive_price_rows": int(((df[["open","high","low","close"]] <= 0).any(axis=1)).sum()),
        "null_rows": int(df.isna().any(axis=1).sum()),
    }
    return rep

raw = load_ohlcv(CFG.data_path)
qc = qc_report(raw)
qc

In [ ]:
# ===== 3. Resampling =====
def resample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    out = df.resample(rule, label="right", closed="right").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
    })
    out = out.dropna()
    return out

m15 = resample_ohlcv(raw, "15min")
h1 = resample_ohlcv(raw, "1h")
print("M1:", len(raw), "M15:", len(m15), "H1:", len(h1))

In [ ]:
# ===== 4. Features =====
def ema(s: pd.Series, span: int) -> pd.Series:
    return s.ewm(span=span, adjust=False).mean()

def add_common_features(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["range"] = x["high"] - x["low"]
    x["body"] = x["close"] - x["open"]
    x["body_abs"] = x["body"].abs()
    x["upper_wick"] = x["high"] - x[["open", "close"]].max(axis=1)
    x["lower_wick"] = x[["open", "close"]].min(axis=1) - x["low"]
    x["body_ratio"] = np.where(x["range"] > 0, x["body_abs"] / x["range"], 0.0)
    x["close_pos"] = np.where(x["range"] > 0, (x["close"] - x["low"]) / x["range"], 0.5)

    for n in [1, 3, 5, 10]:
        x[f"ret_{n}"] = x["close"].pct_change(n)

    x["ema20"] = ema(x["close"], 20)
    x["ema50"] = ema(x["close"], 50)
    x["ema20_slope"] = x["ema20"].diff(3)
    x["ema50_slope"] = x["ema50"].diff(3)

    x["roll_high"] = x["high"].rolling(CFG.rolling_level_n).max().shift(1)
    x["roll_low"] = x["low"].rolling(CFG.rolling_level_n).min().shift(1)

    prev_close = x["close"].shift(1)
    tr = pd.concat([
        x["high"] - x["low"],
        (x["high"] - prev_close).abs(),
        (x["low"] - prev_close).abs()
    ], axis=1).max(axis=1)
    x["atr"] = tr.rolling(CFG.atr_period).mean()

    # session tags on UTC
    hour = x.index.hour
    x["session"] = np.select(
        [
            (hour >= 0) & (hour < 7),
            (hour >= 7) & (hour < 13),
            (hour >= 13) & (hour < 17),
            (hour >= 12) & (hour < 16),
        ],
        ["asia", "london", "ny", "ny_overlap"],
        default="other"
    )
    return x

m15f = add_common_features(m15)
h1f = add_common_features(h1)

# align H1 context using last fully closed H1 bar only
ctx_cols = ["close", "ema20", "ema50", "ema20_slope", "ema50_slope", "atr"]
ctx = h1f[ctx_cols].copy()
ctx.columns = [f"h1_{c}" for c in ctx.columns]
m15f = pd.merge_asof(
    m15f.sort_index(),
    ctx.sort_index(),
    left_index=True,
    right_index=True,
    direction="backward",
    allow_exact_matches=True,
)
m15f.tail(3)

In [ ]:
# ===== 5. Event families =====
def detect_events(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()

    # context regime
    x["trend_up"] = (x["h1_ema20"] > x["h1_ema50"]) & (x["h1_ema20_slope"] > 0)
    x["trend_dn"] = (x["h1_ema20"] < x["h1_ema50"]) & (x["h1_ema20_slope"] < 0)

    # sweep + reclaim
    x["ev_sweep_reclaim_long"] = (
        x["trend_up"] &
        (x["low"] < x["roll_low"]) &
        (x["close"] > x["roll_low"]) &
        (x["close_pos"] > 0.5)
    )
    x["ev_sweep_reclaim_short"] = (
        x["trend_dn"] &
        (x["high"] > x["roll_high"]) &
        (x["close"] < x["roll_high"]) &
        (x["close_pos"] < 0.5)
    )

    # rejection
    x["ev_rejection_long"] = (
        x["trend_up"] &
        (x["lower_wick"] > x["body_abs"] * 1.25) &
        (x["close_pos"] > 0.6)
    )
    x["ev_rejection_short"] = (
        x["trend_dn"] &
        (x["upper_wick"] > x["body_abs"] * 1.25) &
        (x["close_pos"] < 0.4)
    )

    # compression -> expansion
    compress = x["range"].rolling(5).mean() < x["atr"] * 0.75
    x["ev_compression_expansion_long"] = (
        x["trend_up"] &
        compress.shift(1).fillna(False) &
        (x["range"] > x["atr"] * 1.2) &
        (x["close"] > x["open"])
    )
    x["ev_compression_expansion_short"] = (
        x["trend_dn"] &
        compress.shift(1).fillna(False) &
        (x["range"] > x["atr"] * 1.2) &
        (x["close"] < x["open"])
    )

    # quarantined family
    x["ev_momentum_break_long"] = False
    x["ev_momentum_break_short"] = False
    return x

m15e = detect_events(m15f)
event_cols = [c for c in m15e.columns if c.startswith("ev_")]
m15e[event_cols].sum().sort_values(ascending=False)

In [ ]:
# ===== 6. Helpers =====
def backward_swing_stop(df: pd.DataFrame, i: int, side: str, lookback: int) -> float:
    lo = max(0, i - lookback)
    window = df.iloc[lo:i]
    if len(window) == 0:
        return np.nan
    if side == "long":
        return float(window["low"].min())
    return float(window["high"].max())

def apply_atr_floor_cap(entry: float, stop: float, atr: float, side: str) -> float:
    if not CFG.use_atr_floor_cap or pd.isna(atr):
        return stop
    dist = abs(entry - stop)
    if CFG.atr_floor_mult is not None:
        dist = max(dist, CFG.atr_floor_mult * atr)
    if CFG.atr_cap_mult is not None:
        dist = min(dist, CFG.atr_cap_mult * atr)
    return entry - dist if side == "long" else entry + dist

def session_allowed(session_value: str) -> bool:
    return session_value in set(CFG.enabled_sessions)

def families_enabled() -> set:
    return set(CFG.enabled_families)

In [ ]:
# ===== 7. Signal generation =====
def generate_signals(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    fams = families_enabled()

    mapping = [
        ("sweep_reclaim", "long", "ev_sweep_reclaim_long"),
        ("sweep_reclaim", "short", "ev_sweep_reclaim_short"),
        ("rejection", "long", "ev_rejection_long"),
        ("rejection", "short", "ev_rejection_short"),
        ("compression_expansion", "long", "ev_compression_expansion_long"),
        ("compression_expansion", "short", "ev_compression_expansion_short"),
    ]

    for i in range(1, len(df) - (1 + CFG.entry_delay_bars)):
        row = df.iloc[i]
        if not session_allowed(row["session"]):
            continue

        for fam, side, ev_col in mapping:
            if fam not in fams:
                continue
            if not bool(row[ev_col]):
                continue

            entry_ix = i + 1 + CFG.entry_delay_bars
            if entry_ix >= len(df):
                continue

            entry_ts = df.index[entry_ix]
            entry = float(df.iloc[entry_ix]["open"])

            stop = backward_swing_stop(df, i, side, CFG.swing_lookback)
            stop = apply_atr_floor_cap(entry, stop, row["atr"], side)
            if pd.isna(stop) or stop == entry:
                continue

            risk = entry - stop if side == "long" else stop - entry
            if risk <= 0:
                continue

            tp = entry + CFG.tp_r_multiple * risk if side == "long" else entry - CFG.tp_r_multiple * risk

            rows.append({
                "signal_ts": df.index[i],
                "entry_ts": entry_ts,
                "family": fam,
                "side": side,
                "entry": entry,
                "stop": stop,
                "tp": tp,
                "risk": risk,
                "timeout_bars": CFG.timeout_bars,
                "session": row["session"],
            })
    sig = pd.DataFrame(rows)
    return sig

signals = generate_signals(m15e)
signals.head(), signals.shape

In [ ]:
# ===== 8. Execution engine =====
def execute_signals(df: pd.DataFrame, sig: pd.DataFrame) -> pd.DataFrame:
    trades = []
    active_until = None

    for _, s in sig.sort_values("entry_ts").iterrows():
        entry_loc = df.index.get_indexer([s["entry_ts"]])[0]
        if entry_loc < 0:
            continue

        if CFG.one_position_at_a_time and active_until is not None and s["entry_ts"] <= active_until:
            continue

        entry = float(s["entry"])
        stop = float(s["stop"])
        tp = float(s["tp"])
        side = s["side"]

        if side == "long":
            entry_eff = entry + CFG.spread_points + CFG.slippage_points
        else:
            entry_eff = entry - CFG.spread_points - CFG.slippage_points

        exit_reason = "timeout"
        exit_price = float(df.iloc[min(entry_loc + CFG.timeout_bars, len(df)-1)]["close"])
        exit_ts = df.index[min(entry_loc + CFG.timeout_bars, len(df)-1)]

        max_ix = min(entry_loc + CFG.timeout_bars, len(df)-1)
        for j in range(entry_loc, max_ix + 1):
            bar = df.iloc[j]
            ts = df.index[j]

            # conservative tie -> SL first
            if side == "long":
                hit_sl = bar["low"] <= stop
                hit_tp = bar["high"] >= tp
                if hit_sl and hit_tp:
                    exit_reason, exit_price, exit_ts = "sl", stop, ts
                    break
                if hit_sl:
                    exit_reason, exit_price, exit_ts = "sl", stop, ts
                    break
                if hit_tp:
                    exit_reason, exit_price, exit_ts = "tp", tp, ts
                    break
            else:
                hit_sl = bar["high"] >= stop
                hit_tp = bar["low"] <= tp
                if hit_sl and hit_tp:
                    exit_reason, exit_price, exit_ts = "sl", stop, ts
                    break
                if hit_sl:
                    exit_reason, exit_price, exit_ts = "sl", stop, ts
                    break
                if hit_tp:
                    exit_reason, exit_price, exit_ts = "tp", tp, ts
                    break

        risk = abs(entry_eff - stop)
        pnl = (exit_price - entry_eff) if side == "long" else (entry_eff - exit_price)
        r_mult = pnl / risk if risk > 0 else np.nan

        trade = dict(s)
        trade.update({
            "exit_ts": exit_ts,
            "exit_price": exit_price,
            "exit_reason": exit_reason,
            "holding_bars": int(df.index.get_indexer([exit_ts])[0] - entry_loc),
            "r_mult": r_mult,
        })
        trades.append(trade)
        active_until = exit_ts

    return pd.DataFrame(trades)

trades = execute_signals(m15e, signals)
trades.head(), trades.shape

In [ ]:
# ===== 9. Metrics =====
def compute_metrics(trades: pd.DataFrame) -> Dict[str, float]:
    if trades.empty:
        return {
            "total_trades": 0,
            "win_rate": np.nan,
            "profit_factor": np.nan,
            "expectancy_r": np.nan,
            "avg_r": np.nan,
            "median_r": np.nan,
            "sum_r": np.nan,
            "max_drawdown_r": np.nan,
            "max_losing_streak": np.nan,
            "avg_holding_bars": np.nan,
        }

    r = trades["r_mult"].astype(float)
    wins = r[r > 0].sum()
    losses = -r[r < 0].sum()
    pf = wins / losses if losses > 0 else np.inf

    eq = r.cumsum()
    dd = eq - eq.cummax()
    losing = (r <= 0).astype(int)
    max_ls = 0
    cur = 0
    for x in losing:
        cur = cur + 1 if x else 0
        max_ls = max(max_ls, cur)

    return {
        "total_trades": int(len(trades)),
        "win_rate": float((r > 0).mean()),
        "profit_factor": float(pf),
        "expectancy_r": float(r.mean()),
        "avg_r": float(r.mean()),
        "median_r": float(r.median()),
        "sum_r": float(r.sum()),
        "max_drawdown_r": float(dd.min()),
        "max_losing_streak": int(max_ls),
        "avg_holding_bars": float(trades["holding_bars"].mean()),
    }

overall_metrics = compute_metrics(trades)
overall_metrics

In [ ]:
# ===== 10. Diagnostics =====
def add_splits(trades: pd.DataFrame) -> pd.DataFrame:
    if trades.empty:
        return trades.copy()
    x = trades.sort_values("entry_ts").copy()
    n = len(x)
    tr = int(n * CFG.train_ratio)
    va = int(n * (CFG.train_ratio + CFG.val_ratio))
    x["split"] = "test"
    x.iloc[:tr, x.columns.get_loc("split")] = "train"
    x.iloc[tr:va, x.columns.get_loc("split")] = "val"
    return x

trades = add_splits(trades)
trades["year"] = pd.to_datetime(trades["entry_ts"]).dt.year

yearly_metrics = (
    trades.groupby("year", as_index=False)
    .apply(lambda g: pd.Series(compute_metrics(g)))
    .reset_index(drop=True)
)

session_metrics = (
    trades.groupby("session", as_index=False)
    .apply(lambda g: pd.Series(compute_metrics(g)))
    .reset_index(drop=True)
)

family_metrics = (
    trades.groupby("family", as_index=False)
    .apply(lambda g: pd.Series(compute_metrics(g)))
    .reset_index(drop=True)
)

split_metrics = (
    trades.groupby("split", as_index=False)
    .apply(lambda g: pd.Series(compute_metrics(g)))
    .reset_index(drop=True)
)

overall_metrics, yearly_metrics, session_metrics, family_metrics, split_metrics

In [ ]:
# ===== 11. Exports =====
with open(OUTPUT_DIR / "qc_report.json", "w") as f:
    json.dump(qc, f, indent=2)

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump({"overall_metrics": overall_metrics}, f, indent=2)

with open(OUTPUT_DIR / "run_manifest.json", "w") as f:
    json.dump({
        "config": asdict(CFG),
        "enabled_families": list(CFG.enabled_families),
        "enabled_sessions": list(CFG.enabled_sessions),
    }, f, indent=2)

trades.to_csv(OUTPUT_DIR / "trades.csv", index=False)
yearly_metrics.to_csv(OUTPUT_DIR / "yearly_metrics.csv", index=False)
session_metrics.to_csv(OUTPUT_DIR / "session_metrics.csv", index=False)
family_metrics.to_csv(OUTPUT_DIR / "family_metrics.csv", index=False)
split_metrics.to_csv(OUTPUT_DIR / "split_metrics.csv", index=False)

print("Exported to:", OUTPUT_DIR.resolve())

## Suggested test protocol
Jalankan notebook ini dalam 3 mode:

### Mode 1 — Baseline production candidate
Biarkan default apa adanya.

### Mode 2 — Honest broker-cost stress
Isi `spread_points` dan `slippage_points` dari broker / log MT5 riil.  
Jangan pakai angka asumsi jika sudah punya data.

### Mode 3 — Forward-test freeze
Bekukan semua parameter. Jangan ubah rule lagi.  
Hanya kumpulkan hasil forward-test dan audit distribusi.

## Next upgrade path
Jika v1 lolos forward-test, baru lanjut:
1. M5 confirmation layer
2. ATR floor/cap calibration
3. Meta-labeling round 2
